# After the lab: explore the next steps
**RSNA 2026 · Lab 08 · Complementary notebook**

This notebook runs independently: no variables or files from the main notebook are required.
Run the setup below with the hospital URL and your classroom token. It repeats the original one-case workflow
and prepares the same small cohort; the code is collapsed because those steps were explained in the main lab.

Planned hospital access: through **March 2, 2027**. Arena is a separate external service with its own availability
and quota. Its optional inference is disabled by default. Clinical records are fictional; image attribution
is supplied with the session materials.

In [ ]:
#@title Setup 1: original workflow step (run in order)
import os
import subprocess
import sys
if os.environ.get("RSNA_SKIP_INSTALL") != "1":
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                          "httpx==0.28.1", "pydicom==3.0.1", "numpy==2.2.5", "pillow==11.2.1", "pandas==2.2.3", "matplotlib==3.10.1"])

import base64
import getpass
import io
import json
import re
import html
import pandas as pd
from pathlib import Path
import httpx
import pydicom
from IPython.display import display

In [ ]:
#@title Setup 2: original workflow step (run in order)
BASE_URL = os.environ.get("RSNA_BASE_URL", "https://rsnalearninglab.eduardofarina.com").rstrip("/")
TOKEN = os.environ.get("RSNA_TOKEN") or getpass.getpass("Classroom token: ")
client = httpx.Client(headers={"Authorization": f"Bearer {TOKEN}"}, timeout=120)

response = client.get(BASE_URL + "/fhir/R4/metadata")
print("HTTP status:", response.status_code)
response.raise_for_status()
print("FHIR version:", response.json()["fhirVersion"])

In [ ]:
#@title Setup 3: original workflow step (run in order)
from urllib.parse import urljoin, urlparse

def get(path, **kwargs):
    url = urljoin(BASE_URL + "/", path)
    if urlparse(url)[:2] != urlparse(BASE_URL)[:2]:
        raise ValueError("Unexpected server address")
    response = client.get(url, **kwargs)
    response.raise_for_status()
    return response

In [ ]:
#@title Setup 4: original workflow step (run in order)
bundle = get("/fhir/R4/Patient", params={"name": "John"}).json()
print("Matches:", bundle["total"])
patient = bundle["entry"][0]["resource"]
print(json.dumps({key: patient[key] for key in ["id", "name", "birthDate", "gender"]}, indent=2))
patient_id = patient["id"]

In [ ]:
#@title Setup 5: original workflow step (run in order)
bundle = get("/fhir/R4/ImagingStudy", params={"patient": patient_id, "modality": "CR"}).json()
studies = [entry["resource"] for entry in bundle.get("entry", [])]
assert studies, "No CR studies found for this patient"
print([(study["id"], study["started"]) for study in studies])
study = studies[0]  # Change the index to explore another examination.

In [ ]:
#@title Setup 6: original workflow step (run in order)
order = get("/fhir/R4/" + study["basedOn"][0]["reference"]).json()
encounter = get("/fhir/R4/" + order["encounter"]["reference"]).json()
assert order["subject"] == encounter["subject"] == study["subject"]
print("Visit reason:", encounter["reasonCode"][0]["text"])
print("Exam indication:", order["reasonCode"][0]["text"])

In [ ]:
#@title Setup 7: original workflow step (run in order)
def age_at_study(birth_date, study_date):
    """Calculate completed years at acquisition, retaining missing/partial dates."""
    from datetime import date

    if not birth_date or len(birth_date) != 10:
        return None
    birth = date.fromisoformat(birth_date)
    acquired = date.fromisoformat(study_date[:10])
    if acquired < birth:
        raise ValueError("Study date precedes birth date")
    return acquired.year - birth.year - ((acquired.month, acquired.day) < (birth.month, birth.day))

context = {
    "age_years": age_at_study(patient.get("birthDate"), study["started"]),
    "administrative_gender": patient.get("gender", "unknown"),
    "reason_for_visit": encounter["reasonCode"][0]["text"],
}
print(context)

In [ ]:
#@title Setup 8: original workflow step (run in order)
def fhir_report_rows(reports):
    """Translate standard FHIR fields to Imon's original exercise columns."""
    rows = []
    for report in reports:
        accession = next(
            (i["value"] for i in report.get("identifier", []) if i.get("system", "").endswith("/accession")),
            "",
        )
        modality = next(
            (
                c["code"]
                for cat in report.get("category", [])
                for c in cat.get("coding", [])
                if c.get("system") == "http://dicom.nema.org/resources/ontology/DCM"
            ),
            "",
        )
        rows.append(
            {
                "report_id": report["id"],
                "patient_id": report.get("subject", {}).get("reference", "").removeprefix("Patient/"),
                "accession_number": accession,
                "service_request_id": next(
                    (
                        r["reference"].removeprefix("ServiceRequest/")
                        for r in report.get("basedOn", [])
                        if r.get("reference", "").startswith("ServiceRequest/")
                    ),
                    "",
                ),
                "encounter_id": report.get("encounter", {}).get("reference", "").removeprefix("Encounter/"),
                "report_datetime": report.get("issued", report.get("effectiveDateTime")),
                "modality": modality,
                "exam_name": report.get("code", {}).get("text", ""),
                "report_text": base64.b64decode(report["presentedForm"][0]["data"]).decode(),
            }
        )
    return pd.DataFrame(
        rows,
        columns=[
            "report_id",
            "patient_id",
            "accession_number",
            "service_request_id",
            "encounter_id",
            "report_datetime",
            "modality",
            "exam_name",
            "report_text",
        ],
    )

def evaluate_report_matches(study, reports):
    """
    Compare candidate radiology reports against a selected ImagingStudy.
    """

    candidates = reports[reports["patient_id"] == study["patient_id"]].copy()

    # Convert times
    study_time = pd.to_datetime(study["study_datetime"], utc=True)

    candidates["report_datetime"] = pd.to_datetime(candidates["report_datetime"], errors="coerce", utc=True)

    # --------------------------------------------------------
    # Matching criteria
    # --------------------------------------------------------

    candidates["accession_match"] = candidates["accession_number"].fillna("").astype(str) == str(
        study["accession_number"]
    )

    candidates["order_match"] = candidates["service_request_id"].fillna("").astype(str) == str(
        study["service_request_id"]
    )

    candidates["encounter_match"] = candidates["encounter_id"].fillna("").astype(str) == str(
        study["encounter_id"]
    )

    candidates["modality_match"] = candidates["modality"].fillna("").astype(str) == str(study["modality"])

    # --------------------------------------------------------
    # Temporal distance
    # --------------------------------------------------------

    candidates["time_difference_hours"] = (
        candidates["report_datetime"] - study_time
    ).abs().dt.total_seconds() / 3600

    # --------------------------------------------------------
    # Define the high-confidence match
    # --------------------------------------------------------

    candidates["matched"] = (
        candidates["accession_match"]
        & bool(str(study["accession_number"] or "").strip())
        & bool(str(study["service_request_id"] or "").strip())
        & candidates["order_match"]
    )

    # Sort best candidates first
    candidates = candidates.sort_values(
        by=[
            "matched",
            "accession_match",
            "order_match",
            "encounter_match",
            "modality_match",
            "time_difference_hours",
        ],
        ascending=[False, False, False, False, False, True],
    )

    return candidates

def classify_match(row):
    if row["matched"]:
        return "MATCH"

    elif row["encounter_match"] and row["modality_match"]:
        return "POSSIBLE DISTRACTOR"

    else:
        return "NON-MATCH"

In [ ]:
#@title Setup 9: original workflow step (run in order)
# Fetch every page of candidate reports from the RIS via FHIR.
page = get("/fhir/R4/DiagnosticReport", params={"patient": patient_id, "_count": 100}).json()
reports = []
while True:
    reports.extend(entry["resource"] for entry in page.get("entry", []))
    next_url = next((link["url"] for link in page.get("link", []) if link["relation"] == "next"), None)
    if not next_url:
        break
    page = get(next_url).json()
selected_study = {
    "patient_id": patient_id,
    "study_uid": next(i["value"].removeprefix("urn:oid:") for i in study["identifier"] if i["system"] == "urn:dicom:uid"),
    "accession_number": next(i["value"] for i in order["identifier"] if i["system"].endswith("/accession")),
    "service_request_id": order["id"], "encounter_id": encounter["id"],
    "study_datetime": study["started"], "modality": study["modality"][0]["code"]}
candidate_reports = fhir_report_rows(reports)
print("Selected study:", selected_study)
display(candidate_reports[["report_id", "exam_name", "accession_number", "service_request_id"]])

In [ ]:
#@title Setup 10: original workflow step (run in order)
match_results = evaluate_report_matches(selected_study, candidate_reports)
match_results["match_status"] = match_results.apply(classify_match, axis=1)
display_cols = ["report_id", "accession_match", "order_match", "encounter_match", "modality_match", "time_difference_hours", "match_status"]
display(match_results[display_cols])
matched_ids = set(match_results.loc[match_results["matched"], "report_id"])
matching = [r for r in reports if r["id"] in matched_ids
            and r["status"] in ("final", "amended", "corrected")
            and {"reference": "ImagingStudy/" + study["id"]} in r.get("imagingStudy", [])
            and {"reference": "ServiceRequest/" + order["id"]} in r.get("basedOn", [])]
assert matching, "No validated finalized match: skip and review this case"
latest = max(r["issued"] for r in matching)
matching = [r for r in matching if r["issued"] == latest]
assert len(matching) == 1, "Ambiguous finalized matches: review rather than selecting the first row"
report = matching[0]
assert report["subject"] == study["subject"]
report_text = base64.b64decode(report["presentedForm"][0]["data"]).decode("utf-8")
print("Validated report:", report["id"])
print(report_text)

In [ ]:
#@title Setup 11: original workflow step (run in order)
def findings_only(text):
    """Extract the labeled findings section; reject unlabeled reports for review."""
    match = re.search(
        r"(?ims)^\s*findings(?:\s*\(preliminary\))?\s*:\s*(.*?)"
        r"(?=^\s*(?:impression|conclusion|recommendations|addendum)\s*:|\Z)",
        text,
    )
    if not match or not match.group(1).strip():
        raise ValueError("No labeled findings section; review this report manually")
    return " ".join(match.group(1).split())

def extract_report_section(text, section="FINDINGS"):
    """Imon's section extraction, retaining organ subheadings within Findings."""
    if text is None or pd.isna(text):
        return None
    headings = r"EXAM|CLINICAL HISTORY|HISTORY|COMPARISON|TECHNIQUE|FINDINGS|IMPRESSION|CONCLUSION|RECOMMENDATIONS|ADDENDUM|BREAST COMPOSITION|BI-RADS"
    # Require a colon or newline on terminating headings, while permitting inline headings.
    pattern = (
        rf"(?:^|\n)[ \t]*{re.escape(section)}(?:[ \t]*\(preliminary\))?[ \t]*(?::|\n)[ \t\r\n]*"
        rf"(.*?)(?=\n[ \t]*(?:{headings})[ \t]*(?::|\n)|\Z)"
    )
    match = re.search(pattern, str(text), flags=re.IGNORECASE | re.DOTALL)
    return match.group(1).strip() if match else None

def clean_report_text(text):
    if text is None or pd.isna(text):
        return None
    text = html.unescape(str(text)).replace("\xa0", " ")
    return re.sub(r"\s+", " ", text).strip()

sections = {name: extract_report_section(report_text, name)
            for name in ["CLINICAL HISTORY", "COMPARISON", "TECHNIQUE", "FINDINGS", "IMPRESSION"]}
target = clean_report_text(sections["FINDINGS"])
assert target, "Missing findings: review this report"
assert target == findings_only(report_text), "Review extraction differences"
print("Model target:", target)

In [ ]:
#@title Setup 12: original workflow step (run in order)
def modality_extraction_examples(reports):
    """Use three unchanged Imon reports for text-only comparison, never image pairing."""
    examples = []
    for modality, source_id in (("CXR", "RPT-2006"), ("CT", "RPT-2001"), ("Mammography", "RPT-2004")):
        matches = [
            r
            for r in reports
            if any(
                i.get("system", "").endswith("/ris-source-report") and i.get("value") == source_id
                for i in r.get("identifier", [])
            )
        ]
        if len(matches) != 1:
            raise ValueError(f"Expected one original RIS report for {source_id}; review available records")
        resource = matches[0]
        text = base64.b64decode(resource["presentedForm"][0]["data"]).decode()
        examples.append(
            {
                "modality": modality,
                "source_report_id": source_id,
                "fhir_report_id": resource["id"],
                "original_report": text,
                "findings": clean_report_text(extract_report_section(text, "FINDINGS")),
                "impression": clean_report_text(extract_report_section(text, "IMPRESSION")),
                "breast_composition": clean_report_text(extract_report_section(text, "BREAST COMPOSITION")),
                "bi_rads": clean_report_text(extract_report_section(text, "BI-RADS")),
            }
        )
    return examples

if globals().get("USE_FHIR_RECOVERY", False):
    print("Text comparison needs the live RIS. Continue with the recovered CXR target below.")
else:
    extraction_examples = modality_extraction_examples(reports)
    for example in extraction_examples:
        print("\n---", example["modality"], example["source_report_id"], "(text only) ---")
        print(example["original_report"])
        display(pd.DataFrame([
            {"section": key, "extracted_text": example[key]}
            for key in ["findings", "impression", "breast_composition", "bi_rads"]
        ]))
    print("Compared original RIS reports: CXR, CT, Mammography")
    assert target == clean_report_text(extract_report_section(report_text, "FINDINGS"))
    print("Continue with the matched CXR:", study["id"], report["id"])

In [ ]:
#@title Setup 13: original workflow step (run in order)
study_uid = next(item["value"].removeprefix("urn:oid:") for item in study["identifier"]
                 if item["system"] == "urn:dicom:uid")
series_path = f"/dicom-web/studies/{study_uid}/series"
series_list = get(series_path, headers={"Accept": "application/dicom+json"}).json()
print("Series found:", len(series_list))
series_uid = series_list[0]["0020000E"]["Value"][0]
instances_path = f"{series_path}/{series_uid}/instances"
instances = get(instances_path, headers={"Accept": "application/dicom+json"}).json()
instance_uid = instances[0]["00080018"]["Value"][0]
print("Instances in this series:", len(instances))

In [ ]:
#@title Setup 14: original workflow step (run in order)
#@title Helper: open the DICOM response envelope (run once)
from email.parser import BytesParser
from email.policy import default

def read_dicom(response):
    content_type = response.headers.get("content-type", "")
    if content_type.startswith("multipart/"):
        envelope = BytesParser(policy=default).parsebytes(
            ("Content-Type: " + content_type + "\r\nMIME-Version: 1.0\r\n\r\n").encode()
            + response.content)
        parts = [p.get_payload(decode=True) for p in envelope.iter_parts()
                 if p.get_content_type() == "application/dicom"]
        if len(parts) != 1:
            raise ValueError("Expected one DICOM attachment")
        data = parts[0]
    else:
        data = response.content
    return pydicom.dcmread(io.BytesIO(data))

In [ ]:
#@title Setup 15: original workflow step (run in order)
response = get(f"{instances_path}/{instance_uid}",
               headers={"Accept": 'multipart/related; type="application/dicom"; transfer-syntax=*'})
ds = read_dicom(response)
assert str(ds.PatientID) == patient["identifier"][0]["value"]
assert str(ds.AccessionNumber) == selected_study["accession_number"]
assert str(ds.StudyInstanceUID) == study_uid
assert str(ds.SeriesInstanceUID) == series_uid
assert str(ds.SOPInstanceUID) == instance_uid
print("Pixel array:", ds.pixel_array.shape)
print("Photometric interpretation:", ds.PhotometricInterpretation)
print("Window center / width:", ds.get("WindowCenter", "not supplied"), ds.get("WindowWidth", "not supplied"))

In [ ]:
#@title Setup 16: original workflow step (run in order)
#@title Helper: prepare a grayscale radiograph (run once; expand to inspect)
def prepare_cxr(ds, max_side=896, window=None):
    """Single-frame grayscale teaching pipeline; window=(center,width) is optional."""
    import numpy as np
    from PIL import Image
    from pydicom.pixels import apply_modality_lut, apply_voi_lut

    if max_side < 1:
        raise ValueError("max_side must be positive")
    raw = ds.pixel_array
    if raw.ndim != 2 or ds.PhotometricInterpretation not in ("MONOCHROME1", "MONOCHROME2"):
        raise ValueError("This exercise expects a single-frame grayscale radiograph")
    valid = np.isfinite(raw)
    if "PixelPaddingValue" in ds:
        low = float(ds.PixelPaddingValue)
        high = float(ds.get("PixelPaddingRangeLimit", low))
        valid &= (raw < min(low, high)) | (raw > max(low, high))
    pixels = apply_modality_lut(raw, ds).astype(float)
    valid &= np.isfinite(pixels)
    if not valid.any():
        raise ValueError("No usable pixels")
    if window is not None:
        center, width = window
        if not np.isfinite([center, width]).all() or width <= 0:
            raise ValueError("Window width must be positive and values finite")
        # Explicit linear teaching window, distinct from DICOM's LINEAR formula.
        low, high = center - width / 2, center + width / 2
        method = "manual linear window"
    elif "VOILUTSequence" in ds or ("WindowCenter" in ds and "WindowWidth" in ds):
        pixels = apply_voi_lut(pixels, ds).astype(float)
        if "VOILUTSequence" in ds:
            low, high = 0, 2 ** int(ds.VOILUTSequence[0].LUTDescriptor[2]) - 1
        else:
            # pydicom windowing output spans the modality-transformed bit range.
            bits = int(ds.BitsStored)
            low, high = (
                (0, 2**bits - 1) if ds.PixelRepresentation == 0 else (-(2 ** (bits - 1)), 2 ** (bits - 1) - 1)
            )
            if "ModalityLUTSequence" in ds:
                low, high = 0, 2 ** int(ds.ModalityLUTSequence[0].LUTDescriptor[2]) - 1
            else:
                slope, intercept = float(ds.get("RescaleSlope", 1)), float(ds.get("RescaleIntercept", 0))
                low, high = sorted((low * slope + intercept, high * slope + intercept))
        method = "DICOM VOI LUT/window (first preset)"
    else:
        low, high = np.percentile(pixels[valid], [0.5, 99.5])
        method = "0.5–99.5 percentile fallback"
    pixels = np.clip((pixels - low) / max(float(high - low), 1e-8), 0, 1)
    if ds.PhotometricInterpretation == "MONOCHROME1":
        pixels = 1 - pixels
    pixels[~valid] = 0
    image = Image.fromarray(np.rint(pixels * 255).astype("uint8"))
    original_size = image.size
    image.thumbnail((max_side, max_side), Image.Resampling.LANCZOS)
    return image, {
        "original_size": original_size,
        "output_size": image.size,
        "intensity_method": method,
        "max_side": max_side,
    }

In [ ]:
#@title Setup 17: original workflow step (run in order)
MAX_SIDE = 896
image, preprocessing = prepare_cxr(ds, max_side=MAX_SIDE)
print(preprocessing)
display(image)

In [ ]:
#@title Setup 18: original workflow step (run in order)
def make_prompt(context):
    age = context["age_years"] if context["age_years"] is not None else "unknown"
    return ("Draft the findings section of this chest X-ray report. Do not include an impression.\n"
            f"Age at examination: {age} years.\n"
            f"Administrative gender: {context['administrative_gender']}.\n"
            f"Reason for visit: {context['reason_for_visit']}.")

output_dir = Path("cxr-training")
output_dir.mkdir(exist_ok=True)
image_path = output_dir / f"{study['id']}.png"
image.save(image_path)
record = {"patient_id": patient_id, "study_id": study["id"], "report_id": report["id"],
          "study_uid": study_uid, "accession_number": order["identifier"][0]["value"],
          "service_request_id": order["id"], "encounter_id": encounter["id"],
          "match_method": "accession+order+FHIR study",
          "match_status": "RECOVERY" if globals().get("USE_FHIR_RECOVERY", False) else "HIGH_CONFIDENCE",
          "image_path": str(image_path), "context": context,
          "prompt": make_prompt(context), "target": target, "preprocessing": preprocessing}
Path("image-report-pair.json").write_text(json.dumps(record, indent=2))
print("IMAGE:")
from PIL import Image
with Image.open(record["image_path"]) as saved_image:
    display(saved_image.copy())
print("INPUT:\n" + record["prompt"])
print("\nTARGET:\n" + record["target"])

In [ ]:
#@title Setup 19: original workflow step (run in order)
#@title Helper: repeat the same workflow for selected studies

def search_all(resource, **params):
    page = get("/fhir/R4/" + resource, params=params).json()
    records = []
    while True:
        records.extend(entry["resource"] for entry in page.get("entry", []))
        next_url = next((link["url"] for link in page.get("link", []) if link["relation"] == "next"), None)
        if not next_url:
            return records
        page = get(next_url).json()


def build_case(study_id, allow_unreported=False):
    study = get("/fhir/R4/ImagingStudy/" + study_id).json()
    if not any(m["code"] in ("CR", "DX") for m in study["modality"]):
        raise ValueError("Not a chest radiograph case in this pack")
    patient = get("/fhir/R4/" + study["subject"]["reference"]).json()
    order_ref = study["basedOn"][0]["reference"]
    order = get("/fhir/R4/" + order_ref).json()
    encounter = get("/fhir/R4/" + order["encounter"]["reference"]).json()
    assert order["subject"] == encounter["subject"] == study["subject"]
    reports = search_all("DiagnosticReport", patient=patient["id"])
    reports = [r for r in reports if r["status"] in ("final", "corrected", "amended")
               and {"reference": "ImagingStudy/" + study_id} in r.get("imagingStudy", [])
               and {"reference": order_ref} in r.get("basedOn", [])
               and r["subject"] == study["subject"]]
    if not reports and not allow_unreported:
        raise ValueError("No finalized report")
    reports.sort(key=lambda r: r["issued"], reverse=True)
    if len(reports) > 1 and reports[0]["issued"] == reports[1]["issued"]:
        raise ValueError("Ambiguous latest reports: review before pairing")
    report = reports[0] if reports else None
    target = findings_only(base64.b64decode(report["presentedForm"][0]["data"]).decode()) if report else None
    context = {"age_years": age_at_study(patient.get("birthDate"), study["started"]),
               "administrative_gender": patient.get("gender", "unknown"),
               "reason_for_visit": next((r["text"] for r in encounter.get("reasonCode", []) if r.get("text")), "unknown")}
    uid = next(i["value"].removeprefix("urn:oid:") for i in study["identifier"] if i["system"] == "urn:dicom:uid")
    series = get(f"/dicom-web/studies/{uid}/series", headers={"Accept": "application/dicom+json"}).json()
    assert len(series) == 1, "Review series selection"
    series_uid = series[0]["0020000E"]["Value"][0]
    path = f"/dicom-web/studies/{uid}/series/{series_uid}/instances"
    instances = get(path, headers={"Accept": "application/dicom+json"}).json()
    assert len(instances) == 1, "Review instance selection"
    instance_uid = instances[0]["00080018"]["Value"][0]
    ds = read_dicom(get(path + "/" + instance_uid, headers={"Accept": 'multipart/related; type="application/dicom"; transfer-syntax=*'}))
    assert str(ds.StudyInstanceUID) == uid and str(ds.SeriesInstanceUID) == series_uid
    assert str(ds.SOPInstanceUID) == instance_uid
    assert str(ds.PatientID) in {item["value"] for item in patient.get("identifier", [])}
    assert str(ds.AccessionNumber) in {item["value"] for item in order.get("identifier", [])}
    image, preprocessing = prepare_cxr(ds, max_side=MAX_SIDE)
    path = output_dir / f"{study_id}.png"
    image.save(path)
    return {"patient_id": patient["id"], "study_id": study_id, "report_id": report["id"] if report else None,
            "study_uid": uid, "accession_number": order["identifier"][0]["value"],
            "service_request_id": order["id"], "encounter_id": encounter["id"],
            "match_method": "FHIR study+order", "match_status": "HIGH_CONFIDENCE" if report else "NO_REPORT",
            "image_path": str(path), "context": context, "prompt": make_prompt(context),
            "target": target, "preprocessing": preprocessing}

In [ ]:
#@title Setup 20: original workflow step (run in order)
SELECTED_STUDIES = ["study-001", "study-002", "study-003", "study-004"]
records, skipped, all_cases = [], [], []
for study_id in SELECTED_STUDIES:
    try:
        case = build_case(study_id, allow_unreported=True)
        all_cases.append(case)
        if case["target"]:
            records.append(case)
        else:
            skipped.append({"study_id": study_id, "reason": "No finalized report"})
    except ValueError as error:
        skipped.append({"study_id": study_id, "reason": str(error)})
Path("cxr-training.jsonl").write_text("".join(json.dumps(row) + "\n" for row in records))
print("Training examples:", len(records))
print("Skipped:", skipped)

## A. Optional window and resolution experiments

**Try it:** change `MAX_SIDE` to 448 and compare visible detail.
For an optional contrast experiment, set `MANUAL_WINDOW` to `(center, width)` in modality-transformed pixel units.
The manual experiment does not replace the default training image.

In [ ]:
MANUAL_WINDOW = None  # Example after inspecting your pixels: (128, 160)
if MANUAL_WINDOW is not None:
    alternative, details = prepare_cxr(ds, max_side=MAX_SIDE, window=MANUAL_WINDOW)
    print(details)
    display(alternative)

## 13. A tiny data loader — our handoff to model training
A loader opens each saved image and pairs it with the prompt and target.
This iterator needs no training framework. A later model-specific processor/collator will turn it into tensors.
Keep all examinations from the same patient in one split; remove duplicate images before evaluation.
Do not treat this tiny demonstration cohort as a model benchmark.

In [ ]:
from PIL import Image

def training_examples(rows):
    for row in rows:
        with Image.open(row["image_path"]) as image:
            yield {"image": image.convert("RGB"), "prompt": row["prompt"], "target": row["target"]}

example = next(training_examples(records))
print("Image size:", example["image"].size)
print("Prompt:", example["prompt"])
print("Target:", example["target"])

## B. Duplicate cases and patient-level splits
Study-024 deliberately repeats the pixels from study-001 under another examination of the same patient.
Inspect the comparison below. In a larger dataset, compare original pixels before normalization and resizing;
this exact pixel comparison is specific to our planted example.

Keep all studies from one patient in one split. Detect source-image duplicates as well: changing identifiers
does not make copied pixels independent. The small hash-based split below illustrates grouping, not a
statistically meaningful evaluation cohort.

In [ ]:
import hashlib as split_hashlib
import numpy as np

duplicate = build_case("study-024")
original = next(row for row in records if row["study_id"] == "study-001")
with Image.open(original["image_path"]) as a, Image.open(duplicate["image_path"]) as b:
    print("Same prepared pixels:", np.array_equal(np.asarray(a), np.asarray(b)))
print("Same patient:", original["patient_id"] == duplicate["patient_id"])

def patient_split(patient_id):
    bucket = int(split_hashlib.sha256(patient_id.encode()).hexdigest()[:8], 16) % 10
    return "test" if bucket == 0 else "validation" if bucket == 1 else "train"

for row in [original, duplicate]:
    print(row["study_id"], patient_split(row["patient_id"]))

## 14. From training data to deployment: what happens to the next CXR?
Imagine that a new radiograph arrives in the PACS. We want to prepare an **AI draft** while the examination
waits for radiologist review.

```text
Every minute: query FHIR → find CXR with no report → retrieve DICOM → prepare PNG
                                                                        ↓
Radiologist review queue ← save AI draft + provenance ← Radiology Arena inference
```

A **scheduler** asks the hospital for new work at regular intervals (polling).
A **listener** receives an event when an examination arrives. Both can call the same processing function.
Our hospital has no event subscription service, so we demonstrate polling.

For this exercise, “unread” means **an available CXR with no linked active DiagnosticReport**.
It does not prove that nobody has opened the examination. A preliminary report also excludes a study.
In a real RIS, use its reading/worklist status. There is no universal FHIR `unread=true` filter.
Our pack's CR/DX studies are all chest radiographs; a real cohort also needs a chest procedure/body-site filter.

We compare `ImagingStudy` and `DiagnosticReport` records, following every page. A report can link through the
study or the order. This intentionally simple full scan is fine for our tiny hospital.

In [ ]:
def unreported_cxr(studies, reports):
    """In this curated pack CR/DX are CXR. Any active report excludes a study."""
    active = [r for r in reports if r.get("status") not in ("cancelled", "entered-in-error")]
    result = []
    for study in studies:
        if study.get("status") != "available":
            continue
        if not any(m.get("code") in ("CR", "DX") for m in study.get("modality", [])):
            continue
        study_ref = "ImagingStudy/" + study["id"]
        order_refs = {r["reference"] for r in study.get("basedOn", [])}
        linked = any(
            study_ref in {r["reference"] for r in report.get("imagingStudy", [])}
            or bool(order_refs & {r["reference"] for r in report.get("basedOn", [])})
            for report in active
        )
        if not linked:
            result.append(study)
    return result


def find_unreported():
    studies = search_all("ImagingStudy")
    reports = search_all("DiagnosticReport")
    return unreported_cxr(studies, reports)

pending = find_unreported()
print("CXR examinations without a report:", [study["id"] for study in pending])

### Prepare the unreported examination
The missing-report example that we skipped during training is now exactly what we want for inference.
We retrieve its image and clinical context using the same calls as before. **There is no target report.**

In [ ]:
#@title Helper: prepare one unreported CXR using the earlier FHIR and DICOMweb steps
import hashlib

def prepare_unreported(study):
    patient = get("/fhir/R4/" + study["subject"]["reference"]).json()
    order = get("/fhir/R4/" + study["basedOn"][0]["reference"]).json()
    encounter = get("/fhir/R4/" + order["encounter"]["reference"]).json()
    assert order["subject"] == encounter["subject"] == study["subject"]
    context = {"age_years": age_at_study(patient.get("birthDate"), study["started"]),
               "administrative_gender": patient.get("gender", "unknown"),
               "reason_for_visit": next((r["text"] for r in encounter.get("reasonCode", []) if r.get("text")), "unknown")}
    uid = next(i["value"].removeprefix("urn:oid:") for i in study["identifier"] if i["system"] == "urn:dicom:uid")
    series = get(f"/dicom-web/studies/{uid}/series", headers={"Accept": "application/dicom+json"}).json()
    assert len(series) == 1, "Review series selection"
    series_uid = series[0]["0020000E"]["Value"][0]
    path = f"/dicom-web/studies/{uid}/series/{series_uid}/instances"
    instances = get(path, headers={"Accept": "application/dicom+json"}).json()
    assert len(instances) == 1, "Review instance selection"
    instance_uid = instances[0]["00080018"]["Value"][0]
    ds = read_dicom(get(path + "/" + instance_uid, headers={"Accept": 'multipart/related; type="application/dicom"; transfer-syntax=*'}))
    assert str(ds.StudyInstanceUID) == uid and str(ds.SeriesInstanceUID) == series_uid
    assert str(ds.SOPInstanceUID) == instance_uid
    image, preprocessing = prepare_cxr(ds, max_side=MAX_SIDE)
    image_path = Path("inference") / f"{study['id']}.png"
    image_path.parent.mkdir(exist_ok=True)
    image.save(image_path)
    return {"study_id": study["id"], "study_uid": uid, "image_path": str(image_path),
            "image_sha256": hashlib.sha256(image_path.read_bytes()).hexdigest(),
            "context": context, "preprocessing": preprocessing}

In [ ]:
inference_case = prepare_unreported(pending[0]) if pending else None
if inference_case:
    print("Ready for inference:", inference_case["study_id"])
    print("Context kept locally:", inference_case["context"])
    display(Image.open(inference_case["image_path"]))
else:
    print("No unreported CXR right now.")

## 15. Send the image to Radiology Arena
The verified public [Arena API](https://eduardofarina-radiology-arena.hf.space/gradio_api/info)
provides `/generate_single`. It takes an image, model ID, prompt policy and anonymous session ID.
Use **canonical** for its fixed findings-only instruction.

**Current integration boundary:** this API accepts PNG/JPEG, not DICOM, and does **not** accept clinical
context or a custom prompt. We keep age/gender/visit reason locally; this inference is image-only.
A future context-aware deployment needs an inference endpoint that accepts our prepared prompt.
The selected model is pretrained; we have not trained or deployed our own model in this lesson.

Set `RUN_ARENA = True` to upload one prepared teaching image and run inference. The default dry run keeps
“Run all” from consuming shared GPU capacity. Use only the reviewed teaching images for this public service.
Do not send the hospital token, patient identifiers or reference reports.
A sleeping backend, a queue or exhausted quota can prevent inference; record the error instead of inventing a result.

In [ ]:
RUN_ARENA = os.environ.get("RSNA_RUN_ARENA") == "1"  # Set True for the live demonstration.
ARENA_MODEL = "chexone"
if RUN_ARENA and os.environ.get("RSNA_SKIP_INSTALL") != "1":
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gradio-client==2.4.0"])

In [ ]:
#@title Helpers: persistent work tracking and the verified Arena API (run once)
import sqlite3
import uuid

class DraftLedger:
    """Persist claims before inference. Uncertain/failed attempts need manual review."""

    def __init__(self, path):
        self.path = str(path)
        Path(path).parent.mkdir(parents=True, exist_ok=True)
        with sqlite3.connect(self.path) as db:
            db.execute("CREATE TABLE IF NOT EXISTS jobs (job_key TEXT PRIMARY KEY, status TEXT, result TEXT)")
            db.execute("CREATE TABLE IF NOT EXISTS settings (key TEXT PRIMARY KEY, value TEXT)")
            db.execute("INSERT OR IGNORE INTO settings VALUES ('browser_token', ?)", (str(uuid.uuid4()),))

    def browser_token(self):
        with sqlite3.connect(self.path) as db:
            return db.execute("SELECT value FROM settings WHERE key='browser_token'").fetchone()[0]

    def claim(self, job_key):
        with sqlite3.connect(self.path) as db:
            result = db.execute("INSERT OR IGNORE INTO jobs VALUES (?, 'running', NULL)", (job_key,))
            return result.rowcount == 1

    def finish(self, job_key, status, result):
        if status not in ("draft", "needs_review", "skipped"):
            raise ValueError("Unsupported job state")
        with sqlite3.connect(self.path) as db:
            db.execute(
                "UPDATE jobs SET status=?, result=? WHERE job_key=?", (status, json.dumps(result), job_key)
            )

    def rows(self):
        with sqlite3.connect(self.path) as db:
            return [
                {"job_key": key, "status": status, "result": json.loads(result) if result else None}
                for key, status, result in db.execute("SELECT job_key,status,result FROM jobs")
            ]


def arena_draft(result):
    """Parse the verified generate_single Markdown wrapper; reject preview/error responses."""
    if not isinstance(result, (tuple, list)) or len(result) != 2 or not isinstance(result[0], str):
        raise ValueError("Arena response schema changed; review before retrying")
    markdown = result[0]
    if "### Could not generate" in markdown or "[Interface preview" in markdown:
        raise ValueError("Arena returned an error or preview, not a model draft: " + markdown[:500])
    marker = "\n\n---\n\n**Prompt source:**"
    if not markdown.startswith("### ") or marker not in markdown:
        raise ValueError("Arena response wrapper changed; inspect the response")
    body = markdown.split(marker, 1)[0]
    if "\n\n" not in body:
        raise ValueError("Arena returned no report text")
    draft = html.unescape(body.split("\n\n", 1)[1]).strip()
    if not draft:
        raise ValueError("Arena returned an empty report")
    return {
        "draft_text": draft,
        "arena_markdown": markdown,
        "status": "awaiting_radiologist_review",
        "clinical_context_sent": False,
        "prompt_policy": "canonical",
    }


def submit_to_arena(image_path, model_id, browser_token, timeout=300):
    """Upload a PNG, using a separate client with no hospital token or report text."""
    from gradio_client import Client, handle_file

    arena = Client("eduardofarina/radiology-arena", verbose=False, httpx_kwargs={"timeout": 120})
    job = arena.submit(
        image=handle_file(str(image_path)),
        model_id=model_id,
        selected_prompt_policy="canonical",
        browser_token=browser_token,
        api_name="/generate_single",
    )
    # Timeout may mean the backend still runs. The ledger prevents automatic resubmission.
    return arena_draft(job.result(timeout=timeout))

The actual inference call inside the helper is:

```python
arena.submit(
    image=handle_file(image_path),
    model_id="chexone",
    selected_prompt_policy="canonical",
    browser_token=anonymous_session_id,
    api_name="/generate_single",
)
```

The anonymous session ID stays stable across polls; it is separate from the hospital credential.
The server returns display Markdown plus that session ID. We save the raw response and extracted draft,
and reject explicit error/preview messages. The output still requires radiologist review.

In [ ]:
#@title Helper: process at most one new study per polling pass
ledger = DraftLedger(Path("inference") / "jobs.sqlite")

def poll_once(run_inference=False):
    pending = find_unreported()
    if not run_inference:
        return {"mode": "dry_run", "unreported_studies": [s["id"] for s in pending]}
    for study in pending:
        case = prepare_unreported(study)
        job_key = f"{case['study_uid']}|{case['image_sha256']}|{ARENA_MODEL}|canonical-v2"
        if not ledger.claim(job_key):
            continue  # Already attempted: do not charge the GPU again on the next poll.
        try:
            # A radiologist may have reported the study since our first query.
            if study["id"] not in {s["id"] for s in find_unreported()}:
                ledger.finish(job_key, "skipped", {"reason": "Report appeared before submission"})
                continue
            draft = submit_to_arena(case["image_path"], ARENA_MODEL, ledger.browser_token())
            draft.update({"study_id": case["study_id"], "model_id": ARENA_MODEL,
                          "image_sha256": case["image_sha256"], "preprocessing": case["preprocessing"],
                          "report_appeared_during_inference": study["id"] not in {s["id"] for s in find_unreported()}})
            ledger.finish(job_key, "draft", draft)
            return draft
        except Exception as error:
            result = {"study_id": study["id"], "status": "needs_review", "error": str(error)}
            ledger.finish(job_key, "needs_review", result)
            return result
    return {"status": "no_new_work"}

In [ ]:
deployment_result = poll_once(run_inference=RUN_ARENA)
if "draft_text" in deployment_result:
    print("AI DRAFT — awaiting radiologist review\n")
    print(deployment_result["draft_text"])
else:
    print(deployment_result)

## 16. What would a scheduler or listener look like?
A scheduler calls `poll_once` every minute. Run this **bounded** example only if you want to see repeat polling:
the notebook stops after three passes. A listener would call the same worker after an image-arrival event.

A small SQLite ledger records a claim **before** inference and saves the draft afterward. Repeat polls and
notebook restarts using the same file do not resubmit that job. Failed or interrupted attempts stay recorded
for manual inspection: a timeout does not tell us whether remote inference finished. This is not an exactly-once
execution guarantee.

The notebook stores drafts locally and never creates a finalized hospital report. A deployment would run the
worker outside Colab, keep its state on persistent storage and route drafts to a radiologist's review queue.
Keep the hospital read-only token separate from inference credentials. Colab runtimes and their local files are
not durable deployment infrastructure. For a larger hospital, use a paginated incremental scan or supported
FHIR/PACS events plus a queue, with explicit retries and a reconciliation scan.

In [ ]:
RUN_SCHEDULER_DEMO = False
if RUN_SCHEDULER_DEMO:
    import time
    for tick in range(3):
        print("Polling pass", tick + 1, poll_once(run_inference=RUN_ARENA))
        if tick < 2:
            time.sleep(60)

print("Tracked inference jobs:", [{"status": row["status"]} for row in ledger.rows()])
client.close()

## What you can now explain
- A token authorizes requests; a route identifies the information we want.
- FHIR links the patient, visit, order, study and report.
- QIDO finds images; WADO retrieves DICOM objects.
- DICOM display transforms and aspect-preserving resizing prepare pixels.
- Findings are a separate target; impressions never enter the prompt.
- A polling worker can find unreported studies and save inference drafts for review without repeated submission.

**Next session extension:** connect this iterator to a selected VLM processor and benchmark an optional Colab QLoRA run.

### References
[Radiology Arena source and interface](https://huggingface.co/spaces/eduardofarina/radiology-arena),
[HL7 FHIR Patient](https://hl7.org/fhir/R4/patient-definitions.html),
[ServiceRequest](https://hl7.org/fhir/R4/servicerequest.html),
[DICOMweb](https://www.dicomstandard.org/using/dicomweb),
[pydicom pixel processing](https://pydicom.github.io/pydicom/stable/guides/user/working_with_pixel_data.html).
See the accompanying instructor guide and image-source attribution.